# HFP Dil Modelleme Ablasyon Deneyi (WikiText-2)

**Amaç:** TinyShakespeare'deki aşırı ezberlemeyi (overfitting) aşmak ve hangi HFP bileşeninin (cubic, delta, dpfp) dil modellemede gerçekten işe yaradığını görmek için, 10 kat daha büyük olan **WikiText-2** veri setiyle ablasyon yapıyoruz.

Bu notebook, aynı LM görevinde farklı HFP konfigürasyonlarını test eder:

| Koşu | decay | write | fmap | Ne test ediyor? |
|---|---|---|---|---|
| A | exp | additive | elu | Sade HFP (baseline) |
| B | exp | additive | dpfp | dpfp'nin LM etkisi |
| C | exp | delta | dpfp | delta'nın LM etkisi |
| D | cubic | delta | dpfp | Tam reçete (Ek 15) |
| E | cubic | additive | dpfp | cubic'in LM etkisi |

Her biri 3 seed × 1 LR (5e-4) = **15 koşu**. Optimizasyona `weight_decay=0.1` eklendi.

> **Platform:** Google Colab (T4 GPU) veya Kaggle.
> Runtime → Change runtime type → T4 GPU seçin.

In [ ]:
# --- KURULUM ---
import os, subprocess, sys

if os.path.exists('/content'):
    BASE = '/content'
    print("Google Colab ortami")
elif os.path.exists('/kaggle'):
    BASE = '/kaggle/working'
    print("Kaggle ortami")
else:
    BASE = '.'
    print("Yerel ortam")

REPO = os.path.join(BASE, 'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/kayra-hn/HFP.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.40'], check=True)

os.chdir(REPO)
sys.path.insert(0, REPO)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} - {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print("Kurulum tamam!")

In [ ]:
# --- WRAPPER SCRIPT OLUSTUR ---
import os
WRAPPER = os.path.join(REPO, '_ablation_train.py')

script = r'''
import argparse, os, sys, math, csv, random
import numpy as np
import torch
import urllib.request
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from hfp.models.configuration_hfp import HFPConfig
from hfp.models.modeling_hfp import HFPForCausalLM

def estimate_loss(model, train_data, val_data, eval_iters, seq_length, batch_size, device):
    out = {}
    model.eval()
    for split, data in [("train", train_data), ("val", val_data)]:
        losses = []
        for _ in range(eval_iters):
            ix = torch.randint(len(data) - seq_length, (batch_size,))
            x = torch.stack([torch.from_numpy(data[i:i+seq_length].astype(np.int64)) for i in ix]).to(device)
            y = torch.stack([torch.from_numpy(data[i+1:i+seq_length+1].astype(np.int64)) for i in ix]).to(device)
            with torch.no_grad():
                output = model(x, labels=y)
            losses.append(output.loss.item())
        out[split] = np.mean(losses)
    model.train()
    return out

def generate_sample(model, tokenizer, device, max_new_tokens=30):
    model.eval()
    context = "The meaning of life is"
    input_ids = tokenizer.encode(context, return_tensors="pt").to(device)
    if hasattr(model, "hfp") and hasattr(model.hfp, "bulk_states"):
        for b_state in model.hfp.bulk_states:
            b_state.reset_state()
    generated = model.generate(input_ids, max_new_tokens=max_new_tokens,
                               pad_token_id=tokenizer.eos_token_id,
                               do_sample=True, temperature=0.8, top_k=40)
    out = tokenizer.decode(generated[0].tolist())
    model.train()
    return out

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--decay_mode", type=str, required=True)
    parser.add_argument("--write_rule", type=str, required=True)
    parser.add_argument("--key_feature_map", type=str, required=True)
    parser.add_argument("--seed", type=int, required=True)
    parser.add_argument("--lr", type=float, default=5e-4)
    parser.add_argument("--max_iters", type=int, default=5000)
    parser.add_argument("--seq_length", type=int, default=256)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--patience", type=int, default=7)
    parser.add_argument("--eval_interval", type=int, default=200)
    parser.add_argument("--warmup_steps", type=int, default=100)
    parser.add_argument("--log_tag", type=str, required=True)
    args = parser.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained("gpt2")

    print("WikiText-2 verisi indiriliyor (Direct URL)...", flush=True)
    if not os.path.exists("wiki_train.txt"):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt", "wiki_train.txt")
    if not os.path.exists("wiki_valid.txt"):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/valid.txt", "wiki_valid.txt")

    with open("wiki_train.txt", "r", encoding="utf-8") as f:
        train_text = f.read()
    with open("wiki_valid.txt", "r", encoding="utf-8") as f:
        val_text = f.read()

    chunk_size = 100000
    
    train_list = []
    for i in range(0, len(train_text), chunk_size):
        train_list.extend(tokenizer.encode(train_text[i:i+chunk_size], truncation=False))
    train_data = np.array(train_list, dtype=np.int64)

    val_list = []
    for i in range(0, len(val_text), chunk_size):
        val_list.extend(tokenizer.encode(val_text[i:i+chunk_size], truncation=False))
    val_data = np.array(val_list, dtype=np.int64)
    
    print("Train: {:,} | Val: {:,} tokens".format(len(train_data), len(val_data)), flush=True)

    vocab_size = len(tokenizer)
    config = HFPConfig(
        vocab_size=vocab_size, hidden_size=256, num_hidden_layers=4,
        num_attention_heads=4, max_position_embeddings=args.seq_length,
        short_len=8, bulk_dim=32,
        decay_mode=args.decay_mode,
        key_feature_map=args.key_feature_map,
        write_rule=args.write_rule,
        ffn_type="standard",
        rec_block=16
    )
    model = HFPForCausalLM(config).to(device)
    
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.1)
    sch = get_cosine_schedule_with_warmup(opt, args.warmup_steps, args.max_iters)
    
    print("Params: {:.2f}M".format(sum(p.numel() for p in model.parameters()) / 1e6), flush=True)
    print("Config: decay={} write={} fmap={}".format(args.decay_mode, args.write_rule, args.key_feature_map), flush=True)

    best_val_loss = float("inf")
    patience_ctr = 0
    log_file = "{}_log.csv".format(args.log_tag)
    with open(log_file, "w", newline="") as lf:
        csv.writer(lf).writerow(["step", "train_loss", "val_loss", "val_ppl"])

    for it in range(args.max_iters):
        if it % args.eval_interval == 0 or it == args.max_iters - 1:
            losses = estimate_loss(model, train_data, val_data, 50, args.seq_length, args.batch_size, device)
            tl = losses["train"]
            vl = losses["val"]
            val_ppl = math.exp(vl)
            print("Step {}: train {:.4f} val {:.4f} ppl {:.2f}".format(it, tl, vl, val_ppl), flush=True)
            with open(log_file, "a", newline="") as lf:
                csv.writer(lf).writerow([it, tl, vl, val_ppl])
            if vl < best_val_loss:
                best_val_loss = vl
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= args.patience:
                    print("Early stop @ step {}. Best val: {:.4f}".format(it, best_val_loss), flush=True)
                    break

        ix = torch.randint(len(train_data) - args.seq_length, (args.batch_size,))
        x = torch.stack([torch.from_numpy(train_data[i:i+args.seq_length].astype(np.int64)) for i in ix]).to(device)
        y = torch.stack([torch.from_numpy(train_data[i+1:i+args.seq_length+1].astype(np.int64)) for i in ix]).to(device)
        output = model(x, labels=y)
        opt.zero_grad(set_to_none=True)
        output.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sch.step()

    print("DONE best_val_loss={:.4f} best_ppl={:.2f}".format(best_val_loss, math.exp(best_val_loss)), flush=True)

if __name__ == "__main__":
    main()
'''.strip()

with open(WRAPPER, 'w') as f:
    f.write(script)
print(f"Wrapper yazildi: {WRAPPER}")

In [ ]:
# --- 15 ABLASYON KOSUSU ---
import subprocess, sys, os, time

CONFIGS = {
    'A_exp_add_elu':   {'decay': 'exp',                 'write': 'additive', 'fmap': 'elu'},
    'B_exp_add_dpfp':  {'decay': 'exp',                 'write': 'additive', 'fmap': 'dpfp'},
    'C_exp_del_dpfp':  {'decay': 'exp',                 'write': 'delta',    'fmap': 'dpfp'},
    'D_cub_del_dpfp':  {'decay': 'cubic_flux_chunked',  'write': 'delta',    'fmap': 'dpfp'},
    'E_cub_add_dpfp':  {'decay': 'cubic_flux_chunked',  'write': 'additive', 'fmap': 'dpfp'},
}
SEEDS = [0, 1, 2]
LR = 5e-4
MAX_ITERS = 5000
SEQ_LEN = 256
BATCH_SIZE = 16
PATIENCE = 7
EVAL_INTERVAL = 200
WARMUP = 100

all_runs = []
for cfg_name, cfg in CONFIGS.items():
    for seed in SEEDS:
        tag = f"{cfg_name}_s{seed}"
        all_runs.append((cfg_name, cfg, seed, tag))

print(f"{'='*70}")
print(f"  WIKITEXT-2 ABLASYON DENEYI: {len(all_runs)} kosu (5 konfig x 3 seed)")
print(f"{'='*70}")
for cfg_name, cfg, seed, tag in all_runs:
    print(f"  {tag}: {cfg['decay']} + {cfg['write']} + {cfg['fmap']}")
print(f"{'='*70}\n")

total_start = time.time()
for i, (cfg_name, cfg, seed, tag) in enumerate(all_runs):
    print(f"\n[{i+1}/{len(all_runs)}] {tag} basliyor...", flush=True)
    t0 = time.time()
    cmd = [
        sys.executable, WRAPPER,
        '--decay_mode', cfg['decay'],
        '--write_rule', cfg['write'],
        '--key_feature_map', cfg['fmap'],
        '--seed', str(seed),
        '--lr', str(LR),
        '--max_iters', str(MAX_ITERS),
        '--seq_length', str(SEQ_LEN),
        '--batch_size', str(BATCH_SIZE),
        '--patience', str(PATIENCE),
        '--eval_interval', str(EVAL_INTERVAL),
        '--warmup_steps', str(WARMUP),
        '--log_tag', tag
    ]
    
    # Hata detayı görmek için FİLTRESİZ CANLI YAZDIRMA eklendi
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(f"    | {line.strip()}", flush=True)
            
    process.wait()
    elapsed = time.time() - t0
    status = 'OK' if process.returncode == 0 else 'HATA'
    print(f"  [{status}] {elapsed:.0f}s", flush=True)
    if process.returncode != 0:
        print(f"  !!! KOSU HATA VERDI. Sonraki kosuya geciliyor... !!!", flush=True)

total = time.time() - total_start
print(f"\n{'='*70}")
print(f"  TUM {len(all_runs)} KOSU TAMAMLANDI - {total:.0f}s ({total/60:.1f} dk)")
print(f"{'='*70}")

In [ ]:
# --- SONUCLARI TOPLA VE ANALIZ ET ---
import csv, os
import numpy as np

CONFIG_LABELS = {
    'A_exp_add_elu':   'exp + additive + elu',
    'B_exp_add_dpfp':  'exp + additive + dpfp',
    'C_exp_del_dpfp':  'exp + delta + dpfp',
    'D_cub_del_dpfp':  'cubic + delta + dpfp',
    'E_cub_add_dpfp':  'cubic + additive + dpfp',
}
SEEDS = [0, 1, 2]

results = {}
for cfg_name in CONFIG_LABELS:
    for seed in SEEDS:
        tag = f"{cfg_name}_s{seed}"
        log_file = f"{tag}_log.csv"
        if not os.path.exists(log_file):
            print(f"  EKSIK: {log_file}")
            continue
        best_val = float('inf')
        with open(log_file) as f:
            reader = csv.DictReader(f)
            for row in reader:
                vl = float(row['val_loss'])
                if vl < best_val:
                    best_val = vl
        results[tag] = best_val

print(f"Toplam {len(results)} sonuc okundu.\n")

# --- ANA TABLO ---
print(f"{'='*90}")
print(f"  ABLASYON SONUCLARI — Best Validation Loss (WikiText-2)")
print(f"{'='*90}")
print(f"{'Konfigurasyon':<30} {'Seed 0':>8} {'Seed 1':>8} {'Seed 2':>8} {'Ort.':>8} {'Std':>6} {'PPL':>8}")
print('-'*90)

config_means = {}
for cfg_name, label in CONFIG_LABELS.items():
    vals = []
    seed_str = []
    for seed in SEEDS:
        tag = f"{cfg_name}_s{seed}"
        if tag in results:
            v = results[tag]
            vals.append(v)
            seed_str.append(f"{v:.4f}")
        else:
            seed_str.append("N/A")
    if vals:
        mean_v = np.mean(vals)
        std_v = np.std(vals, ddof=1) if len(vals) > 1 else 0
        ppl = np.exp(mean_v)
        config_means[cfg_name] = (mean_v, std_v, ppl, vals)
        print(f"  {label:<28} {seed_str[0]:>8} {seed_str[1]:>8} {seed_str[2]:>8} {mean_v:>8.4f} {std_v:>6.4f} {ppl:>8.1f}")
    else:
        print(f"  {label:<28} {'N/A':>8} {'N/A':>8} {'N/A':>8}")

if config_means:
    best_cfg = min(config_means.items(), key=lambda x: x[1][0])
    print(f"\n  En iyi: {CONFIG_LABELS[best_cfg[0]]} (val loss {best_cfg[1][0]:.4f}, PPL {best_cfg[1][2]:.1f})")

# --- BILESEN KATKI ANALIZI ---
print(f"\n{'='*90}")
print(f"  BILESEN KATKI ANALIZI")
print(f"{'='*90}")

def compare(name, base_cfg, test_cfg, desc):
    if base_cfg in config_means and test_cfg in config_means:
        base_m, _, base_p, _ = config_means[base_cfg]
        test_m, _, test_p, _ = config_means[test_cfg]
        diff_loss = test_m - base_m
        diff_ppl = test_p - base_p
        direction = "IYILESTIRME" if diff_loss < -0.01 else "KOTU" if diff_loss > 0.01 else "NOTR"
        print(f"\n  {name}")
        print(f"    {CONFIG_LABELS[base_cfg]} -> {CONFIG_LABELS[test_cfg]}")
        print(f"    {desc}")
        print(f"    Val loss: {base_m:.4f} -> {test_m:.4f} ({diff_loss:+.4f})")
        print(f"    PPL:      {base_p:.1f} -> {test_p:.1f} ({diff_ppl:+.1f})")
        print(f"    -> {direction}")

compare("[1] dpfp etkisi", 'A_exp_add_elu', 'B_exp_add_dpfp',
        "elu -> dpfp (decay=exp, write=additive sabit)")

compare("[2] delta etkisi", 'B_exp_add_dpfp', 'C_exp_del_dpfp',
        "additive -> delta (decay=exp, fmap=dpfp sabit)")

compare("[3] cubic etkisi (additive+dpfp)", 'B_exp_add_dpfp', 'E_cub_add_dpfp',
        "exp -> cubic (write=additive, fmap=dpfp sabit)")

compare("[4] cubic etkisi (delta+dpfp)", 'C_exp_del_dpfp', 'D_cub_del_dpfp',
        "exp -> cubic (write=delta, fmap=dpfp sabit)")

compare("[5] tam recete vs sade", 'A_exp_add_elu', 'D_cub_del_dpfp',
        "Sade HFP (exp+add+elu) -> Tam recete (cubic+delta+dpfp)")

print(f"\n{'='*90}")
print("  WikiText-2 ablasyon analizi tamamlandi.")
print(f"{'='*90}")